<a href="https://colab.research.google.com/github/yrarjun59/COMFYUI/blob/main/comfyui_colab_Flux.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1 – Install [ComfyUI](https://github.com/comfyanonymous/ComfyUI) repo and install the requirements.

In [ ]:
# ==========================================
# 1. INSTALL COMFYUI + PYTORCH (CUDA 12.4)
# ==========================================

import os
import torch


# Check if PyTorch with CUDA is already installed
if torch.cuda.is_available():
    print(f"✅ PyTorch {torch.__version__} with CUDA already installed. Skipping reinstall.")
else:
    print("📦 Installing PyTorch with CUDA 12.4...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Clone ComfyUI if not present
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI
    print("✅ ComfyUI cloned.")
else:
    print("✅ ComfyUI already exists.")

# Install requirements (skip torch to avoid conflict)
!pip install -q -r /content/ComfyUI/requirements.txt

!apt-get install -y aria2 > /dev/null 2>&1
print("✅ aria2 installed.")


print("✅ Setup complete.")

Cell 2 – Install Custom Nodes (With Existence Checks)

In [ ]:
# ==========================================
# 2. INSTALL CUSTOM NODES
# ==========================================

nodes = {
    "comfyui-manager": "https://github.com/ltdrdata/ComfyUI-Manager.git",
    "RES4LYF": "https://github.com/ClownsharkBatwing/RES4LYF.git",
    "rgthree-comfy": "https://github.com/rgthree/rgthree-comfy.git",
    "ComfyUI-Easy-Use": "https://github.com/yolain/ComfyUI-Easy-Use.git",
}

base = "/content/ComfyUI/custom_nodes"
for name, url in nodes.items():
    path = os.path.join(base, name)
    if not os.path.exists(path):
        print(f"📥 Cloning {name}...")
        !git clone -q {url} {path}
    else:
        print(f"✅ {name} already exists, skipping.")

print("✅ All custom nodes installed.")

📥 Cloning comfyui-manager...
📥 Cloning RES4LYF...
📥 Cloning rgthree-comfy...
📥 Cloning ComfyUI-Easy-Use...
✅ All custom nodes installed.


Cell 3 – Nunchaku Nodes (Separate, with Check)

In [ ]:
# ==========================================
# 3. INSTALL NUNCHAKU NODES
# ==========================================

import os
nunchaku_path = "/content/ComfyUI/custom_nodes/nunchaku_nodes"
if not os.path.exists(nunchaku_path):
    !git clone -q https://github.com/mit-han-lab/ComfyUI-nunchaku {nunchaku_path}
    print("✅ Nunchaku nodes cloned.")
else:
    print("✅ Nunchaku nodes already exist, skipping.")

Cell - 4 Download ALL official models (bypass Xet issues)

In [ ]:
import os
import shutil

LOCAL_BASE = "/content/ComfyUI/models"

# Mapping: (official_repo_id, file_path_in_repo, destination_relative_path)
# These are the ONLY official files you need to download
official_files = [
    # Flux + Qwen3 Base (from Comfy-Org)
    ("Comfy-Org/vae-text-encorder-for-flux-klein-4b",
     "split_files/diffusion_models/flux-2-klein-4b.safetensors",
     "diffusion_models/flux-2-klein-4b.safetensors"),

    ("Comfy-Org/vae-text-encorder-for-flux-klein-4b",
     "split_files/text_encoders/qwen_3_4b.safetensors",
     "text_encoders/qwen_3_4b.safetensors"),

    ("Comfy-Org/vae-text-encorder-for-flux-klein-4b",
     "split_files/vae/flux2-vae.safetensors",
     "vae/flux2-vae.safetensors"),

    # Qwen Image Edit models (from Comfy-Org)
    ("Comfy-Org/Qwen-Image_ComfyUI",
     "split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors",
     "text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors"),

    ("Comfy-Org/Qwen-Image_ComfyUI",
     "split_files/vae/qwen_image_vae.safetensors",
     "vae/qwen_image_vae.safetensors"),

    # Qwen Lightning LoRA (from lightx2v)
    ("lightx2v/Qwen-Image-Edit-2511-Lightning",
     "Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors",
     "loras/Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors"),
]

for repo_id, src_path, dst_rel in official_files:
    dest_full = os.path.join(LOCAL_BASE, dst_rel)

    # Skip if already exists
    if os.path.exists(dest_full):
        print(f"⏭️ Skipping existing: {dst_rel}")
        continue

    # Ensure destination folder exists
    os.makedirs(os.path.dirname(dest_full), exist_ok=True)

    # Build the Hugging Face resolve URL (direct download)
    url = f"https://huggingface.co/{repo_id}/resolve/main/{src_path}"

    print(f"📥 Downloading {dst_rel} from {repo_id}...")

    # Use aria2c for reliable downloading (16 connections, auto-resume, 10 retries)
    !aria2c -x 16 -s 16 -k 1M \
        --retry-wait=5 \
        --max-tries=10 \
        --continue=true \
        --console-log-level=error \
        "{url}" \
        -d "{os.path.dirname(dest_full)}" \
        -o "{os.path.basename(dest_full)}"

    print(f"✅ Downloaded: {dst_rel}")

print("\n🎉 All official models downloaded successfully!")

Cell 6 – Download Specific Models from  HF Repo (The Core)

In [ ]:
import os
from huggingface_hub import HfApi

REPO_ID = "yrarjun/civitai-models"
LOCAL_BASE = "/content/ComfyUI/models"

# Only these folders from your repo
allowed_prefixes = [
    "checkpoints/",
    # "ultralytics/",
    "unet/"
]

api = HfApi()
all_files = api.list_repo_files(REPO_ID)
target_files = [f for f in all_files if any(f.startswith(p) for p in allowed_prefixes)]

print(f"📦 Found {len(target_files)} files to download from your repo.")

for file_path in target_files:
    local_file = os.path.join(LOCAL_BASE, file_path)
    if os.path.exists(local_file):
        print(f"⏭️ Skipping existing: {file_path}")
        continue

    os.makedirs(os.path.dirname(local_file), exist_ok=True)
    url = f"https://huggingface.co/{REPO_ID}/resolve/main/{file_path}"

    print(f"📥 Downloading: {file_path}")
    !wget -c -q --show-progress --tries=3 "{url}" -O "{local_file}"
    print(f"✅ Downloaded: {file_path}")

print("🎉 All repo files downloaded!")

Cell 7 – Verification of Model Integrity (New)

In [ ]:
# ==========================================
# 5. VERIFY MODEL INTEGRITY
# ==========================================

!pip install -q safetensors

import os
import safetensors

base = "/content/ComfyUI/models"
corrupt = []
valid = []

if not os.path.exists(base):
    print("❌ Models folder not found! Run Cell 4 first.")
else:
    for root, dirs, files in os.walk(base):
        for file in files:
            if file.endswith(".safetensors"):
                path = os.path.join(root, file)
                try:
                    with safetensors.safe_open(path, framework="pt", device="cpu") as f:
                        valid.append(path)
                except Exception as e:
                    corrupt.append((path, str(e)))

    print(f"✅ Valid .safetensors files: {len(valid)}")
    if corrupt:
        print("❌ CORRUPT FILES FOUND:")
        for path, err in corrupt:
            print(f"   - {path}\n     Error: {err}")
        print("\n⚠️  Delete and re-download these files before proceeding.")
    else:
        print("🎉 All .safetensors files are clean!")

✅ Valid .safetensors files: 7
🎉 All .safetensors files are clean!


Cell 8 – Launch ComfyUI with ngrok (Unchanged)

In [ ]:
# ==========================================
# 6. LAUNCH COMFYUI WITH NGROK
#    Live Timestamp + Live Uptime
#    Cell stays running
# ==========================================

!pip install -q pyngrok

from pyngrok import ngrok
import subprocess
import socket
import time
from google.colab import userdata
from datetime import datetime
import IPython.display as display

# ------------------------------------------
# Get NGROK token
# ------------------------------------------

NGROK_TOKEN = userdata.get("NGROK_TOKEN")

if not NGROK_TOKEN:
    raise ValueError("Please set NGROK_TOKEN in Colab secrets.")

!ngrok config add-authtoken $NGROK_TOKEN

# ------------------------------------------
# Record launch time
# ------------------------------------------

start_time = datetime.now()

print(
    f"🚀 ComfyUI launch initiated at: "
    f"{start_time.strftime('%Y-%m-%d %H:%M:%S')}"
)

# ------------------------------------------
# Start ComfyUI
# ------------------------------------------

print("▶️ Starting ComfyUI...")

comfy_process = subprocess.Popen([
    "python",
    "/content/ComfyUI/main.py",
    "--dont-print-server"
])

# ------------------------------------------
# Wait for ComfyUI port 8188
# ------------------------------------------

port = 8188

print("⏳ Waiting for ComfyUI server to start...")

while True:

    try:
        sock = socket.create_connection(
            ("127.0.0.1", port),
            timeout=2
        )

        sock.close()

        print("\n")

        print(f"✅ ComfyUI server is running on port {port}")
        print(
            f"   Started at: "
            f"{start_time.strftime('%Y-%m-%d %H:%M:%S')}"
        )

        break

    except OSError:

        elapsed = datetime.now() - start_time

        print(
            f"\r⏳ Waiting... "
            f"{str(elapsed).split('.')[0]} elapsed",
            end="",
            flush=True
        )

        time.sleep(2)

# ------------------------------------------
# Create ngrok tunnel
# ------------------------------------------

print("\n🌐 Creating ngrok tunnel...")

public_url = ngrok.connect(
    port,
    bind_tls=True
)

print("\n" + "=" * 55)
print("🌐 COMFYUI PUBLIC URL")
print("=" * 55)
print(public_url)
print("=" * 55)

# ------------------------------------------
# LIVE SERVER STATUS
# ------------------------------------------

print("\n🟢 ComfyUI + ngrok are running")
print("🔄 Live uptime monitor started")
print("🛑 Interrupt the cell to stop monitoring")
print()

try:

    while True:

        now = datetime.now()
        uptime = now - start_time

        # Clear/update the same area of output
        display.clear_output(wait=True)

        print("=" * 60)
        print("🚀 COMFYUI + NGROK LIVE STATUS")
        print("=" * 60)

        print(
            f"📅 Started: "
            f"{start_time.strftime('%Y-%m-%d %H:%M:%S')}"
        )

        print(
            f"🕐 Current: "
            f"{now.strftime('%Y-%m-%d %H:%M:%S')}"
        )

        print(
            f"⏱️ Uptime: "
            f"{str(uptime).split('.')[0]}"
        )

        print(f"🔌 ComfyUI Port: {port}")

        print(f"🌐 Public URL: {public_url}")

        print("🟢 Status: RUNNING")

        print("=" * 60)

        time.sleep(1)

except KeyboardInterrupt:

    print("\n🛑 Live monitor stopped.")
    print("ComfyUI/ngrok process may still be running.")


### To Downlaod the Folder

In [ ]:
# 1. Zip the folder (Replace 'output.zip' with your preferred name, and check your folder path)
!zip -r output.zip /content/ComfyUI/output/Datasets/Rina

# 2. Download the zipped file to your local machine
from google.colab import files
files.download('output.zip')
